# Unified Hyperparameter Sweep (Optuna) - Keras CAE & PatchCore

This notebook provides a unified entry point for optimizing hyperparameters across all 15 MVTec AD categories for both Keras CAE and PatchCore.

**Cross-Framework Stability**: Combining TensorFlow and PyTorch within the same persistent Jupyter process often causes CUDA driver state conflicts and GPU memory leaks. For optimal stability, this notebook runs the sweep via `scripts/sweep.py --model all` (or individual models via `--model keras` / `--model patchcore`), ensuring complete process isolation and zero memory leakage between trials.

Results are incrementally persisted to:
- `data/hyperparameters/keras_cae_best.json`
- `data/hyperparameters/patchcore_best.json`

In [ ]:
import os
import sys
import warnings
from pathlib import Path

# Suppress noisy deprecation warnings from external libraries
warnings.filterwarnings("ignore", category=FutureWarning, module=".*timm.*")
warnings.filterwarnings("ignore", category=FutureWarning, module=".*anomalib.*")

# Dynamically navigate to repository root (supports both local environment and Google Colab)
current_dir = Path.cwd()
while not (current_dir / "app").exists() and current_dir != current_dir.parent:
    current_dir = current_dir.parent
PROJECT_ROOT = current_dir
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.core.tf_device import preload_cuda_shared_libraries
preload_cuda_shared_libraries()

print(f"Project root resolved: {PROJECT_ROOT.resolve()}")


In [ ]:
# Run unified sweep for BOTH models with isolated subprocess execution
!python scripts/sweep.py --model all


In [ ]:
# Alternatively, run sweeps individually as needed:
# !python scripts/sweep.py --model keras
# !python scripts/sweep.py --model patchcore


In [ ]:
import json

keras_file = PROJECT_ROOT / "data/hyperparameters/keras_cae_best.json"
patchcore_file = PROJECT_ROOT / "data/hyperparameters/patchcore_best.json"

print("=" * 60)
print("HYPERPARAMETER SWEEP RESULTS SUMMARY")
print("=" * 60)

if keras_file.exists():
    k_res = json.loads(keras_file.read_text(encoding="utf-8"))
    print(f"\n[Keras CAE] Completed categories ({len(k_res)}/15): {list(k_res.keys())}")
else:
    print("\n[Keras CAE] No results found yet.")

if patchcore_file.exists():
    p_res = json.loads(patchcore_file.read_text(encoding="utf-8"))
    print(f"\n[PatchCore] Completed categories ({len(p_res)}/15): {list(p_res.keys())}")
else:
    print("\n[PatchCore] No results found yet.")
